# Analyzing asylum denial rates by immigration court
**Julia Ingram / CBS News based on analysis by Joe Yerardi / Philadelphia Inquirer**

This notebook answers several questions about asylum outcomes using data from TRAC.

In [3]:
import pandas as pd
import os

## Import data

Source: `trac_asylum_scraper` outputs in `data/`

In [4]:
# Function to load each CSV and split fymon into separate year and month columns
def load(filename, group_col=None):
    df = pd.read_csv(f"data/exported/{filename}")
    df["year"]  = df["fymon"].str[:4].astype(int)
    df["month"] = df["fymon"].str[5:].astype(int)
    cols = ["court"]
    if group_col:
        cols.append(group_col)
    cols += ["year", "month", "total", "granted", "denied", "other"]
    return df[cols]

total_by_month          = load("master_totals.csv")
custody_by_month        = load("master_custody.csv",        group_col="custody_status")
language_by_month       = load("master_language.csv",       group_col="language")
representation_by_month = load("master_representation.csv", group_col="representation")

## Q1: Overall denial rates

In [5]:
# Helper function to summarize the total granted and denied for a given court or set of courts
def summarize_court(df, court_label):
    c = df[df["court"] == court_label]
    total   = c["total"].sum()
    granted = c["granted"].sum()
    effective_denied = c["denied"].sum() + c["other"].sum()
    return pd.Series({
        "total_cases":    total,
        "total_granted":  granted,
        "total_denied":   effective_denied,
        "pct_granted":    granted / total,
        "pct_denied":     effective_denied / total,
    })
asylum_total = pd.DataFrame({
    "Philadelphia": summarize_court(total_by_month, "Philadelphia"),
    "United States": summarize_court(total_by_month, "United States"),
}).T

asylum_total

,total_cases,total_granted,total_denied,pct_granted,pct_denied
Philadelphia,15505.0,5906.0,9599.0,0.380909,0.619091
United States,989301.0,383324.0,605977.0,0.387470,0.612530


## Q2: Denial rates by presidential administration

In [6]:
# Map each month to the president who was in office
# Assigns January of the transition year to the outgoing president
def assign_president(row):
    y, m = row["year"], row["month"]
    if   (y < 2001) or (y == 2001 and m == 1):  return "Other"    # Clinton / pre-data
    elif (y < 2009) or (y == 2009 and m == 1):  return "Bush"
    elif (y < 2017) or (y == 2017 and m == 1):  return "Obama"
    elif (y < 2021) or (y == 2021 and m == 1):  return "Trump I"
    elif (y < 2025) or (y == 2025 and m == 1):  return "Biden"
    else:                                         return "Trump II"

PRESIDENT_ORDER = ["Bush", "Obama", "Trump I", "Biden", "Trump II"]

asylum_by_president = (
    total_by_month
    .assign(president=lambda d: d.apply(assign_president, axis=1))
    .query("president != 'Other'")
    .groupby(["president", "court"])
    .agg(total=("total", "sum"), denied=("denied", "sum"), other=("other", "sum"))
    .reset_index()
    .assign(effective_denied=lambda d: d["denied"] + d["other"],
            pct_denied=lambda d: (d["denied"] + d["other"]) / d["total"])
    .pivot_table(index="president", columns="court",
                 values=["total", "effective_denied", "pct_denied"])
    .reindex(PRESIDENT_ORDER)
)

asylum_by_president.columns = [f"{col[1].lower().replace(' ', '_')}_{col[0]}" for col in asylum_by_president.columns]
asylum_by_president


,philadelphia_effective_denied,united_states_effective_denied,philadelphia_pct_denied,united_states_pct_denied,philadelphia_total,united_states_total
president,,,,,,
Bush,3564.0,139130.0,0.710668,0.591094,5015.0,235377.0
Obama,760.0,84478.0,0.436280,0.503268,1742.0,167859.0
Trump I,1786.0,141219.0,0.619709,0.690048,2882.0,204651.0
Biden,1814.0,142996.0,0.499449,0.554351,3632.0,257952.0
Trump II,1541.0,93348.0,0.768579,0.808775,2005.0,115419.0


## Q3: Biden's final year vs. Trump's second term so far (2024 vs 2025)

In [7]:
asylum_biden_vs_trump = (
    total_by_month
    .query("year >= 2024")
    .assign(president=lambda d: d["year"].map({2024: "Biden", 2025: "Trump"}))
    .dropna(subset=["president"])
    .groupby(["president", "court"])
    .agg(total=("total", "sum"), denied=("denied", "sum"), other=("other", "sum"))
    .reset_index()
    .assign(effective_denied=lambda d: d["denied"] + d["other"],
            pct_denied=lambda d: (d["denied"] + d["other"]) / d["total"])
    .pivot_table(index="president", columns="court",
                 values=["total", "effective_denied", "pct_denied"])
    .reindex(["Biden", "Trump"])
)

asylum_biden_vs_trump.columns = [f"{col[1].lower().replace(' ', '_')}_{col[0]}" for col in asylum_biden_vs_trump.columns]
asylum_biden_vs_trump

,philadelphia_effective_denied,united_states_effective_denied,philadelphia_pct_denied,united_states_pct_denied,philadelphia_total,united_states_total
president,,,,,,
Biden,466.0,48032.0,0.531963,0.586858,876.0,81846.0
Trump,1601.0,99048.0,0.766762,0.800278,2088.0,123767.0


## Q4: Monthly denial rates since Jan 2024

In [8]:
asylum_by_month_recent = (
    total_by_month
    .query("year >= 2024")
    .assign(month_label=lambda d: d["year"].astype(str) + "-" + d["month"].astype(str).str.zfill(2))
    .groupby(["month_label", "court"])
    .agg(total=("total", "sum"), denied=("denied", "sum"), other=("other", "sum"))
    .reset_index()
    .assign(effective_denied=lambda d: d["denied"] + d["other"],
            pct_denied=lambda d: (d["denied"] + d["other"]) / d["total"])
    .pivot_table(index="month_label", columns="court",
                 values=["total", "effective_denied", "pct_denied"])
    .sort_index()
)

asylum_by_month_recent.columns = [f"{col[1].lower().replace(' ', '_')}_{col[0]}" for col in asylum_by_month_recent.columns]
asylum_by_month_recent

,philadelphia_effective_denied,united_states_effective_denied,philadelphia_pct_denied,united_states_pct_denied,philadelphia_total,united_states_total
month_label,,,,,,
2024-01,28.0,3362.0,0.538462,0.526135,52.0,6390.0
2024-02,32.0,3380.0,0.426667,0.516109,75.0,6549.0
2024-03,33.0,3626.0,0.434211,0.526423,76.0,6888.0
2024-04,41.0,4040.0,0.493976,0.554184,83.0,7290.0
2024-05,24.0,4240.0,0.380952,0.562036,63.0,7544.0
2024-06,37.0,3531.0,0.493333,0.559943,75.0,6306.0
2024-07,29.0,3997.0,0.604167,0.574364,48.0,6959.0
2024-08,47.0,4344.0,0.587500,0.620660,80.0,6999.0
2024-09,40.0,4123.0,0.588235,0.638038,68.0,6462.0


## Q5: Denial rates by custody status

In [9]:
asylum_by_custody = (
    custody_by_month
    .groupby(["custody_status", "court"])
    .agg(total=("total", "sum"), denied=("denied", "sum"), other=("other", "sum"))
    .reset_index()
    .assign(effective_denied=lambda d: d["denied"] + d["other"],
            pct_denied=lambda d: (d["denied"] + d["other"]) / d["total"])
    .pivot_table(index="custody_status", columns="court",
                 values=["total", "effective_denied", "pct_denied"])
)

asylum_by_custody.columns = [f"{col[1].lower().replace(' ', '_')}_{col[0]}" for col in asylum_by_custody.columns]
asylum_by_custody

,philadelphia_effective_denied,united_states_effective_denied,philadelphia_pct_denied,united_states_pct_denied,philadelphia_total,united_states_total
custody_status,,,,,,
detained,800.0,114410.0,0.982801,0.839294,814.0,136317.0
never_detained,7139.0,393482.0,0.591466,0.576574,12070.0,682448.0
not_known,2.0,46.0,1.000000,0.567901,2.0,81.0
released,1658.0,98039.0,0.633066,0.575161,2619.0,170455.0


## Q6: Denial rates by language spoken

In [10]:
asylum_by_language = (
    language_by_month
    .groupby(["language", "court"])
    .agg(total=("total", "sum"), denied=("denied", "sum"), other=("other", "sum"))
    .reset_index()
    .assign(effective_denied=lambda d: d["denied"] + d["other"],
            pct_denied=lambda d: (d["denied"] + d["other"]) / d["total"])
    .pivot_table(index="language", columns="court",
                 values=["total", "effective_denied", "pct_denied"])
)

asylum_by_language.columns = [f"{col[1].lower().replace(' ', '_')}_{col[0]}" for col in asylum_by_language.columns]
asylum_by_language

,philadelphia_effective_denied,united_states_effective_denied,philadelphia_pct_denied,united_states_pct_denied,philadelphia_total,united_states_total
language,,,,,,
english,839.0,45743.0,0.515040,0.571802,1629.0,79998.0
non_english,8760.0,560234.0,0.631306,0.616114,13876.0,909303.0
total,9599.0,605977.0,0.619091,0.612530,15505.0,989301.0


## Q7: Denial rates by access to legal representation

In [11]:
asylum_by_representation = (
    representation_by_month
    .groupby(["representation", "court"])
    .agg(total=("total", "sum"), denied=("denied", "sum"), other=("other", "sum"))
    .reset_index()
    .assign(effective_denied=lambda d: d["denied"] + d["other"],
            pct_denied=lambda d: (d["denied"] + d["other"]) / d["total"])
    .pivot_table(index="representation", columns="court",
                 values=["total", "effective_denied", "pct_denied"])
)

asylum_by_representation.columns = [f"{col[1].lower().replace(' ', '_')}_{col[0]}" for col in asylum_by_representation.columns]
asylum_by_representation

,philadelphia_effective_denied,united_states_effective_denied,philadelphia_pct_denied,united_states_pct_denied,philadelphia_total,united_states_total
representation,,,,,,
not_represented,1876.0,163613.0,0.826068,0.852533,2271.0,191914.0
represented,7723.0,442364.0,0.583573,0.554767,13234.0,797387.0
